# Emulation - Data Space Inversion (DSI)

Everything we have done so far has needed the model. GLM needed it to fill a Jacobian. Monte Carlo needed it once per realisation. iES needed it once per realisation *per iteration*. For the Freyberg model that is fine - it runs in a second. For a real model that takes an hour, it is the whole ballgame.

*Data space inversion* (DSI) takes a different route. Instead of history matching the model, we history match a cheap statistical stand-in - an **emulator** - built from model runs we have already done. The emulator maps the statistical relationships between model outputs: between the outputs we have measurements for, and the outputs we care about forecasting. Conditioning happens in that output ("data") space, not in parameter space.

The consequence is the interesting bit: **DSI needs no new model runs at all.** We already have a prior Monte Carlo ensemble from the iES notebook. That ensemble is all DSI needs.

The catch is not what people usually assume. It is not that you lose access to parameters - you can carry those along too, and we will. It is that the emulator is a *model of the model*: an approximation built from a finite number of runs, and it can only ever be as good as that sample allows. We will get to what that costs.

This notebook follows the same maths and notation as the [intro to EVA and DSI](../part0_intro_to_eva_and_dsi/intro_to_eva_and_dsi.ipynb) notebook in part0, which works through it on a tiny made-up dataset - a good place to start if the algebra below moves too fast. The original paper is [Sun and Durlofsky (2017)](https://doi.org/10.1007/s11004-016-9672-8), with later variants in e.g. [Lima et al (2020)](https://doi.org/10.1007/s10596-020-09933-w). There are also [GMDSI webinars on YouTube](https://www.youtube.com/watch?v=s2g3HaJa1Wk&t=1564s). The [part2 DSI notebook](../part2_10_eva_and_dsi/2_freyberg_ensemble_data_space_inversion.ipynb) does all of this on a much higher-dimensional problem.

## How DSI works, briefly

Notation follows the part0 [intro to EVA and DSI](../part0_intro_to_eva_and_dsi/intro_to_eva_and_dsi.ipynb) notebook, and after it [Lima et al (2020)](https://doi.org/10.1007/s10596-020-09933-w).

Write $\mathbf{d}$ for the vector of *all* model outputs - those corresponding to our measurements and those corresponding to our forecasts, stacked together. A prior Monte Carlo run gives us $N_e$ realisations of it. Its covariance is

$$\mathbf{C}_d = \Delta\mathbf{D}\,\Delta\mathbf{D}^T, \qquad \Delta\mathbf{D} = \frac{1}{\sqrt{N_e-1}}\left[\mathbf{d}_1-\bar{\mathbf{d}},\;\dots,\;\mathbf{d}_{N_e}-\bar{\mathbf{d}}\right]$$

where $\Delta\mathbf{D}$ is the deviations matrix and $\bar{\mathbf{d}}$ the ensemble mean. $\mathbf{C}_d$ is the object that matters: it encodes how every output co-varies with every other - crucially, how the measured quantities co-vary with the forecasts.

Take the SVD of the deviations matrix, $\Delta\mathbf{D} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$, which gives us a square root of the covariance:

$$\mathbf{C}_d^{1/2} = \mathbf{U}\boldsymbol{\Sigma}$$

Now the trick. Instead of parameterising the *model*, DSI parameterises the *outputs*:

$$\mathbf{d}_{\text{PCA}} = \bar{\mathbf{d}} + \mathbf{C}_d^{1/2}\,\mathbf{x}$$

That is the entire emulator. Feed it a vector $\mathbf{x}$, get back a complete set of model outputs. Why is that sensible? Because if $\mathbf{x}\sim N(\mathbf{0},\mathbf{I})$ then

$$\mathrm{Cov}\left[\mathbf{C}_d^{1/2}\mathbf{x}\right] = \mathbf{U}\boldsymbol{\Sigma}\,\mathbf{I}\,\boldsymbol{\Sigma}^T\mathbf{U}^T = \mathbf{U}\boldsymbol{\Sigma}^2\mathbf{U}^T = \mathbf{C}_d$$

The emulator reproduces the prior covariance of the model outputs *exactly*. That is why the elements of $\mathbf{x}$ - the "latent-space parameters" - get a mean of zero and a standard deviation of one, and it is literally what `prepare_pestpp` writes into `dsi.unc` for us later.

History matching is then: find the $\mathbf{x}$ whose measured-quantity entries match our observations. Because $\mathbf{x}\mapsto\mathbf{d}$ is **linear** and the prior on $\mathbf{x}$ is **Gaussian**, this is the linear-Gaussian Bayesian problem - the same one FOSM solves, but posed in output space instead of parameter space. `pestpp-ies` solves it iteratively, which also copes with the mild nonlinearity that transformations introduce later.

Three things to notice, because all three come back:

1. The model appears nowhere in that equation. Once $\mathbf{U}$ and $\boldsymbol{\Sigma}$ are computed, MODFLOW is done.
2. Everything rests on $\mathbf{C}_d$ - a mean and a covariance. Those two moments pin down a distribution **only if that distribution is Gaussian**.
3. $\Delta\mathbf{D}$ has $N_e$ columns, so $\mathbf{C}_d^{1/2}$ has at most $N_e$ useful columns however long $\mathbf{d}$ is. The emulator can only ever represent $N_e$ independent directions of variability. Remember this one.

## The Current Tutorial

In this notebook we will:
1. Reuse the prior Monte Carlo ensemble from the iES notebook as training data - and screen it, because it needs it
2. Build a DSI emulator with `pyemu.emulators.DSI`
3. Condition it with `pestpp-ies`, using the same observations, weights and phi as every other part1 notebook
4. Compare the DSI forecast posterior against the iES forecast posterior
5. Be honest about what DSI cannot do

### Admin

> **This notebook needs the iES notebook to have been run first.** We use its prior observation ensemble as our training data, so run [part1_13](../part1_13_basic_ies/freyberg_ies.ipynb) before this one. We do *not* build or run the model here - that is rather the point.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import time
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;

import pyemu
import flopy
sys.path.insert(0,"..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10
pyemu.plot_utils.font = 10

## The training data

DSI needs an ensemble of model outputs that spans **both** the quantities we have measurements for **and** the forecasts we care about. That is exactly what a prior Monte Carlo run produces, and it is the only time the model has to run.

We already paid for one in the iES notebook: the prior ensemble, iteration 0 of the first `pestpp-ies` run. Let's go and get it.

In [ ]:
# the iES notebook's master directory - we are borrowing its results, not re-running anything
ies_d = os.path.join('..','part1_13_basic_ies','master_ies')
assert os.path.exists(ies_d), "run the part1_13 iES notebook first!"

pst = pyemu.Pst(os.path.join(ies_d,'freyberg_pp.pst'))

# iteration 0 is the prior - the model outputs before any data was assimilated
oe = pd.read_csv(os.path.join(ies_d,'freyberg_pp.0.obs.csv'),index_col=0)
oe.columns = [c.lower() for c in oe.columns]
oe = oe.astype(float)
oe.shape

So we have an ensemble of model outputs: one row per realisation, one column per observation in the control file. Note what is in there and what it cost us:

In [ ]:
print(f'realisations in the training data : {oe.shape[0]}')
print(f'model outputs per realisation     : {oe.shape[1]}')
print(f'non-zero weighted observations    : {pst.nnz_obs}')
print(f'observation groups being fitted    : {sorted(pst.nnz_obs_groups)}')
print(f'forecasts                         : {pst.forecast_names}')
print()
print('new model runs needed for DSI      : 0')

## First, screen the training data

It is tempting to go straight to the emulator. Don't.

DSI works by taking the principal components of the training data. Principal components are driven by **variance**, and variance is not robust: one output with a silly value in one realisation can swamp everything else. And a prior Monte Carlo run on a groundwater model *will* contain silly values - realisations where a cell went dry, or where the solver did not really converge.

Heads in this model are a few tens of metres. Let's see if that is what we actually have.

In [ ]:
head_obs = [o for o in oe.columns if o.startswith('trgw')]

print('range of simulated heads across the whole prior ensemble:')
print(f'  min: {oe[head_obs].values.min():.3g}')
print(f'  max: {oe[head_obs].values.max():.3g}')

Ouch. A massively negative head is not a water level, it is a model that fell over. Let's find out how widespread it is, and which outputs are affected.

In [ ]:
# a negative head is unambiguous garbage - there is no reading of this model in
# which a water level is -1e15 m
negative = (oe[head_obs] < 0).any(axis=1)
print(f'realisations with a negative head: {negative.sum()} of {oe.shape[0]}')

# which sites go bad, and how badly
worst = oe[head_obs].min().sort_values()
print('\nworst five outputs (minimum value over the ensemble):')
print(worst.head(5).to_string())

# the high end is less clear cut - plot it and see. Sorted, so we are looking for
# a step: somewhere the sensible realisations stop and the broken ones start
max_head = oe[head_obs].max(axis=1).sort_values().values
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(range(len(max_head)), max_head, 'b.', ms=5)
ax.axhline(100, color='r', ls='--', lw=1.5, label='100 m - the cut we are about to make')
ax.axhspan(20, 50, color='g', alpha=0.12, label='plausible water levels here')
ax.set_yscale('log')
ax.set_xlabel('realisation (sorted)')
ax.set_ylabel('highest simulated head in that realisation (m, log scale)')
ax.set_title('Is there an obvious line between good and broken?')
ax.legend(fontsize=8)
plt.tight_layout();

No, there isn't. The negative heads are unarguable, but above them the ensemble just tails off smoothly out of the plausible band and keeps going - there is no step, no gap, nowhere the data tells us to draw the line. We will have to choose one, and own it.

First though, the part that matters most. Is any of this contaminating something we actually care about?

In [ ]:
# are the observations we are fitting affected? and the forecasts?
affected = worst[worst < 0].index.tolist()
print('contaminated outputs that are weighted observations:')
print(' ', [o for o in affected if o in pst.nnz_obs_names] or 'none')
print('contaminated outputs that are forecasts:')
print(' ', [o for o in affected if o in pst.forecast_names] or 'none')

It is in both. Twelve of the observations we are *fitting* are contaminated - all at the `trgw-0-26-6` site - and so is `trgw-0-9-1:4383.5`, the groundwater level forecast we have been tracking since the trial-and-error notebook. These are not realisations that are slightly off; they are numerical garbage.

Worth pausing on, because none of it is a DSI problem. It was sitting in the prior ensemble the whole time, quietly skewing the prior histograms back in the Monte Carlo and iES notebooks. DSI just makes it impossible to ignore, because of what variance does to principal components. Here is the damage, measured as the share of total ensemble variance sitting in a single output:

In [ ]:
def cumulative_energy(data):
    """the share of ensemble variance captured by each successive principal
    component - the same calculation DSI uses internally when deciding how
    many components to keep"""
    X = data.astype(float)
    z = (X - X.mean()) / np.sqrt(X.shape[0] - 1)
    s = np.linalg.svd(z.values, full_matrices=False)[1]
    return np.cumsum(s**2) / np.sum(s**2)


def n_components_for(ce, threshold):
    """how many components are needed to reach `threshold` of the variance"""
    return int(np.argmax(ce >= threshold) + 1)


ce_raw = cumulative_energy(oe)
print(f'biggest single output as a share of total variance: '
      f'{100*oe.var().max()/oe.var().sum():.1f}%')
print()
print('components needed, using the raw ensemble:')
for t in [0.9, 0.99, 0.999]:
    print(f'  {t*100:5.1f}% of variance -> {n_components_for(ce_raw, t)} component(s)')

One component holds 99.9% of the "variance". That is not a low-dimensional problem, it is a broken one - the PCA is describing a single blown-up number and nothing else. An emulator fitted to this would be worthless.

So we screen. We drop the offending realisations and keep the rest.

In [ ]:
# drop the unambiguous failures, plus realisations whose heads climb implausibly
# high. The 100 m cut is OUR CHOICE - the plot above shows no break to guide us,
# so this is a judgement about what is physically plausible. Change it and see
MAX_PLAUSIBLE_HEAD = 100.0
blown_up = negative | (oe[head_obs] > MAX_PLAUSIBLE_HEAD).any(axis=1)

data = oe.loc[~blown_up, :]
print(f'training data: {data.shape[0]} realisations (dropped {blown_up.sum()})')
print(f'head range now: {data[head_obs].values.min():.2f} to {data[head_obs].values.max():.2f} m')

ce = cumulative_energy(data)

# how many components does it take to describe the ensemble, before and after?
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4))
ax.plot(np.arange(1, len(ce_raw)+1), ce_raw, 'r-', label='raw ensemble')
ax.plot(np.arange(1, len(ce)+1), ce, 'b-', label='after screening')
ax.axhline(0.99, color='k', ls='--', lw=1.0, label='99% of variance')
ax.set_xlim(0, 40)
ax.set_xlabel('number of principal components')
ax.set_ylabel('cumulative share of variance')
ax.set_title('One bad realisation makes the problem look\none-dimensional when it is not')
ax.legend(fontsize=9)
plt.tight_layout()

print(f'\ncomponents needed for 99% of the variance:'
      f'\n  raw            : {n_components_for(ce_raw, 0.99)}'
      f'\n  after screening: {n_components_for(ce, 0.99)}')

Much healthier. The variance is now spread over many components, which is what we would expect from an ensemble that genuinely varies in a lot of directions.

> **Lesson, and it is the most practical one in this notebook:** an emulator is only ever as good as the ensemble it was trained on. Screening the training data is not optional housekeeping, it is part of the method.

Notice how little of that screening was objective. The negative heads decided themselves. The 100 m cut did not - the ensemble tails off smoothly, so we drew a line based on what we think a water level in this aquifer can plausibly be. Draw it at 60 m and you drop 44 realisations instead of 20; draw it at 200 m and you keep realisations that are probably junk. Like the streamflow weight back in part1_01, it is a subjective call that changes the answer, so make it deliberately and write it down.

We also threw away whole realisations rather than individual outputs. That is another choice - we could have dropped the `trgw-0-24-4` site entirely and kept its realisations. Which is better depends on whether you need that site.

There is a second scale problem hiding underneath this one, though, and screening does not touch it. Look at where the variance actually lives now:

In [ ]:
# how much of the ensemble variance does each observation group carry?
groups = pst.observation_data.groupby('obgnme').obsnme.apply(list)
rows = []
for grp, names in groups.items():
    names = [n for n in names if n in data.columns]
    if len(names) == 0:
        continue
    rows.append((grp, data[names].var().sum()))
vardf = pd.DataFrame(rows, columns=['group','total_variance'])
vardf['share'] = 100 * vardf.total_variance / vardf.total_variance.sum()
vardf = vardf.sort_values('total_variance')

# head sites in blue so they are easy to pick out from the fluxes
colors = ['b' if g.startswith('trgw') else '0.5' for g in vardf.group]

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.barh(vardf.group, vardf.total_variance, color=colors)
ax.set_xscale('log')
ax.set_xlabel('total variance across the training ensemble (log scale)')
ax.set_title('Where the ensemble variance lives\n(blue = head observation sites)')
# label the ones that actually matter
for y, (v, sh) in enumerate(zip(vardf.total_variance, vardf.share)):
    if sh >= 1.0:
        ax.text(v*1.5, y, f'{sh:.0f}%', va='center', fontsize=9)
plt.tight_layout();

Note the log scale - that plot spans five orders of magnitude. Streamflow, travel time and the two exchange fluxes carry essentially all of the variance. Every head site is down at the far left, contributing a fraction of a percent between them - and heads are half of the observations we are actually fitting.

This is not because heads are uninformative. It is because an SVD ranks directions by *absolute* variance and has no idea that one column is in metres and another is in cubic metres per day. Head variance across this ensemble is around $10^3$; streamflow variance is around $10^8$. The components will describe streamflow and ignore heads, whatever we do with weights later.

So the emulator we are about to build is, in effect, a model of the flows. We will come back and fix this once we have seen what else needs fixing.

## Building the emulator

`pyemu.emulators` is the entry point for all things emulation. To build a `DSI` object we need the training data. Optionally we pass a `Pst`, which `DSI` uses later to help build a PEST interface for the emulator, plus transformations and an SVD truncation level.

We will keep `energy_threshold=1.0`, which means no truncation - keep every component. For a problem this size that costs nothing.

In [ ]:
from pyemu.emulators import DSI

dsi = DSI(pst=pst,               # optional; used to build the PEST interface later
          data=data,             # the training data - this is the required bit
          transforms=None,       # optional; see the limitations discussion below
          energy_threshold=1.0)  # 1.0 = keep every component

dsi.fit();

`fit()` did the work: centre the training data, take its SVD, and store the projection that turns a vector of latent coefficients back into a full set of model outputs.

The singular values tell us how much each component carries:

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(9,3.5))
axes[0].plot(dsi.s,'b-')
axes[0].set_yscale('log')
axes[0].set_xlabel('component')
axes[0].set_ylabel('singular value')
axes[0].set_title('singular value spectrum')

axes[1].plot(ce,'b-')
axes[1].axhline(0.99,color='r',ls='--',lw=1.0,label='99% of variance')
axes[1].set_xlabel('number of components')
axes[1].set_ylabel('cumulative share of variance')
axes[1].set_title('cumulative energy')
axes[1].legend()
plt.tight_layout();

## What are the parameters now?

This is the conceptual jump. Our model had 68 adjustable parameters: pilot point hydraulic conductivities and recharge. The emulator has none of them. Its parameters are the elements of $\mathbf{x}$ - coefficients on the principal components of the output ensemble. Following the part0 notebook, you can think of them as a mapping from the model's base parameters onto "super parameters" that drive the surrogate.

To run the emulator, hand it a vector of those coefficients. That is the entire forward run:

In [ ]:
print(f'the emulator takes a vector of {dsi.s.shape[0]} latent coefficients')

pvals = np.random.normal(0, 1, dsi.s.shape)
sim = dsi.predict(pvals)

print(f'and returns {sim.shape[0]} simulated model outputs')
sim.head()

No MODFLOW. No files. A matrix multiply.

Any vector of coefficients gives you a complete, self-consistent set of model outputs - heads, flows, travel time, historic and forecast - in microseconds.

A common thing to say at this point is that DSI has cost us access to parameters, so we can never ask what hydraulic conductivity field produced a given forecast. That is not true, and it is worth being clear about. Nothing in $\bar{\mathbf{d}} + \mathbf{C}_d^{1/2}\mathbf{x}$ cares what the entries of $\mathbf{d}$ *mean* - they are just numbers that co-vary. So we can stack the model's parameters onto the output vector and DSI will learn their covariance along with everything else, and hand us their posterior too. We will do exactly that later in this notebook.

The real constraint is point 3 from the maths section: $N_e$ directions of variability, no matter how many rows $\mathbf{d}$ has. That is a much more interesting problem, and we come back to it once we have a working emulator.

## A PEST interface for the emulator

We now want `pestpp-ies` to adjust those latent coefficients until the emulated outputs match our measurements. That needs a control file, template files and instruction files - and `prepare_pestpp()` builds all of it.

We pass `use_runstor=True`, which sets the emulator up to be driven through PEST++'s **external run manager**. More on why in a moment.

In [ ]:
t_d = 'dsi_template'
pst_dsi = dsi.prepare_pestpp(t_d=t_d, use_runstor=True)

# prepare_pestpp builds a fresh control file, so the pestpp options from the model
# don't come across. We very much want our forecasts to travel with us
pst_dsi.pestpp_options['forecasts'] = pst.pestpp_options['forecasts']
assert pst_dsi.forecast_names == pst.forecast_names

# and it doesn't copy binaries over either - note we need pestpp-ies, but no MODFLOW
hbd.prep_bins(t_d)

sorted(os.listdir(t_d))

`dsi.pickle` is the fitted emulator. `dsi_pars.csv` holds the latent coefficients, `dsi_sim_vals.csv` the emulated outputs, and `forward_run.py` is the "model" - it loads the emulator and calls `predict()`.

Let's see how the interface changed:

In [ ]:
print(f'{"":22s}{"model":>10s}{"emulator":>10s}')
print(f'{"adjustable parameters":22s}{pst.npar_adj:>10d}{pst_dsi.npar_adj:>10d}')
print(f'{"observations":22s}{pst.nobs:>10d}{pst_dsi.nobs:>10d}')
print(f'{"weighted observations":22s}{pst.nnz_obs:>10d}{pst_dsi.nnz_obs:>10d}')

The observations are identical - same 36 weighted observations, same groups, same weights. That is deliberate, and it is what makes this notebook comparable to the ones before it: **phi means the same thing here as it did in the GLM and iES notebooks.** Whatever number we end up with can be read against those.

The parameters are a different animal entirely: latent coefficients instead of pilot points.

### Prior data conflict, and extrapolation

Ask an emulator about a part of output space its training runs never visited and it will answer. It will not warn you, and it will not widen its uncertainty. The maths puts no bound on $\mathbf{x}$, so $\bar{\mathbf{d}} + \mathbf{C}_d^{1/2}\mathbf{x}$ can be pushed anywhere in the span of $\mathbf{C}_d^{1/2}$ - well outside anything the model actually produced. Extrapolation is not impossible; it is *unconstrained*. You get a confidently wrong answer, delivered with the same narrow posterior as a well-supported one. (We will see a vivid example of this shortly.)

The version of this that PEST++ can actually detect is *prior data conflict*: a measured value lying outside the range the training ensemble ever produced for that quantity. There is no $\mathbf{x}$ that reaches it, so `pestpp-ies` will contort the whole ensemble trying. Better to spot those observations and drop them - which is what `ies_drop_conflicts` does. Turn it on; for DSI it should be the default.

Treat a conflict as information, not a nuisance. It is telling you the prior ensemble does not span reality, and the honest fix is more or better model runs, not a louder emulator.

In [ ]:
pst_dsi.pestpp_options["ies_drop_conflicts"] = True

## Conditioning the emulator

Now we run `pestpp-ies` on the emulator instead of on the model.

One new thing here. Normally we deploy a swarm of PANTHER workers, each running the model in its own directory. That is the right design when the model takes minutes, but it is entirely wrong for DSI: the emulator run takes microseconds, so all the time goes into starting python and reading files. Deploying workers would make DSI a hundred times slower than it needs to be.

Instead we use PEST++'s **external run manager**, with the `/e` switch. It works batch-wise: PEST++ writes every parameter set it wants evaluated into a run storage file (`dsi.rns`), calls the forward run **once**, and reads all the results back. So `dsi.predict()` gets called a single time per iteration, on the whole ensemble at once, in one process. No workers, no ports, no worker directories.

In [ ]:
# as always, as many realisations as you can afford. Here that is all of them
pst_dsi.pestpp_options["ies_num_reals"] = data.shape[0]
pst_dsi.control_data.noptmax = 3
pst_dsi.write(os.path.join(t_d,"dsi.pst"),version=2)

# run in a copy, so the template folder stays clean
m_d = 'master_dsi'
if os.path.exists(m_d):
    shutil.rmtree(m_d)
shutil.copytree(t_d, m_d)

start = time.time()
pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=m_d)
print(f'\nconditioning took {time.time()-start:.1f} seconds')

Seconds. For comparison, the iES run in the previous notebook needed 50 model runs per iteration, and the prior Monte Carlo before it needed 250.

Let's look at the objective function. Remember this is the *same* phi - same observations, same weights - that we have been minimising since the trial-and-error notebook.

In [ ]:
pst_dsi = pyemu.Pst(os.path.join(m_d,"dsi.pst"))
phidf = pd.read_csv(os.path.join(m_d,"dsi.phi.actual.csv"))

fig,ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(phidf.iteration, phidf['mean'], "bo-", label='ensemble mean phi')
ax.fill_between(phidf.iteration, phidf['min'], phidf['max'],
                color='b', alpha=0.15, label='ensemble min/max')
ax.axhline(pst_dsi.nnz_obs, color='r', ls='--', lw=1.5,
           label=f'nnz_obs = {pst_dsi.nnz_obs}')
ax.set_yscale('log')
ax.set_xlabel('iteration')
ax.set_ylabel('$\\Phi$')
ax.legend()
plt.tight_layout();

Phi drops hard, and then keeps going - straight past the number of non-zero weighted observations (the red line).

That red line is a rough overfitting alarm. If weights were set as the inverse of the standard deviation of measurement noise, then a model fitting the data *as well as the noise allows* should land at a phi of roughly `nnz_obs`. Going well below it means we are fitting noise.

But look closely at what our weights actually are. Throughout part1 we have used `HEAD_WEIGHT = 1.0` and `SFR_WEIGHT = 0.003` - numbers we chose for balance and visibility back in the trial-and-error notebook, not measurements of anything. They are not inverse noise standard deviations. `pestpp-ies` even told us as much in the record file: with no noise information available it fell back to `ies_no_noise`, so there is no noise floor for phi to respect.

So we cannot actually tell, from this run, whether we are overfitting. That is not a DSI flaw - it is the bill arriving for a subjective weighting choice made twelve notebooks ago. Part2 does this properly, assigning weights as `1/standard_deviation` so that the `nnz_obs` line means something.

## Did it fit?

Same stochastic 1-to-1 plot we have used since the iES notebook, so you can flip between the two. Prior in grey, posterior in blue.

In [ ]:
oe_pr_dsi, oe_pt_dsi = hbd.load_ies_obs_ensembles(m_d, case='dsi')
hbd.plot_1to1_ensemble(pst_dsi, prior=oe_pr_dsi, posterior=oe_pt_dsi,
                       title='DSI emulator');

The emulator fits the measured data well - unsurprising, given the phi history. The question that actually matters is whether it gets the *forecasts* right.

## Predictive uncertainty analysis

This is what we came for. Prior in grey, posterior in blue, and the truth as a dashed black line. Remember we only know the truth because this is a synthetic problem - and remember that a forecast is a success only if the posterior *brackets* the truth, not if it hits it exactly.

In [ ]:
for forecast in pst_dsi.forecast_names:
    fig,ax = plt.subplots(1,1,figsize=(5,2.8))
    oe_pr_dsi.loc[:,forecast].hist(bins=20,alpha=0.5,color="0.5",ax=ax,label='prior')
    oe_pt_dsi.loc[:,forecast].hist(bins=20,alpha=0.5,color="b",ax=ax,label='posterior')
    v = pst_dsi.observation_data.loc[forecast,"obsval"]
    ax.axvline(v,color='k',ls='--',lw=2.0,label='truth')
    ax.set_ylabel('count')
    ax.set_title(forecast)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Hold on - are these answers even possible?

Before we celebrate, let's ask something the fit statistics cannot tell us: are the numbers the emulator produced physically possible at all?

Streamflow in a river cannot be negative. Neither can a particle travel time. Let's count.

In [ ]:
def check_feasible(oe_post, label):
    """count values the emulator produced that cannot physically happen"""
    gage = [n for n in groups['gage-1'] if n in oe_post.columns]
    neg = (oe_post[gage] < 0)
    ptime = oe_post['part_time']
    print(f'[{label}]')
    print(f'  negative streamflow  : {int(neg.values.sum())} values, in '
          f'{int(neg.any(axis=1).sum())} of {oe_post.shape[0]} realisations')
    print(f'  lowest streamflow    : {oe_post[gage].values.min():,.1f} m3/d')
    print(f'  negative travel times: {int((ptime<0).sum())}, '
          f'lowest {ptime.min():,.1f} days')


check_feasible(oe_pt_dsi, 'untransformed')

That is not a rounding problem. A large fraction of our posterior realisations contain rivers flowing backwards at thousands of cubic metres a day, and some contain particles that arrive before they set off.

Every one of those realisations went into the forecast histograms above.

## Why this happens: the normality assumption

Go back to the emulator equation:

$$\mathbf{d}_{\text{PCA}} = \bar{\mathbf{d}} + \mathbf{C}_d^{1/2}\,\mathbf{x}, \qquad \mathbf{x}\sim N(\mathbf{0},\mathbf{I})$$

We showed that this reproduces $\mathbf{C}_d$ exactly. What we glossed over is that a mean and a covariance **do not describe a distribution** - unless that distribution happens to be Gaussian. The multivariate normal is the one family that is completely determined by its first two moments. For anything else, matching the mean and covariance matches two summary statistics and throws the rest away.

So what does DSI actually generate? A linear combination of Gaussian variables is Gaussian. Which means $\mathbf{d}_{\text{PCA}}$ is **multivariate normal, by construction** - regardless of what our prior ensemble looked like. If the outputs really were Gaussian, that is exact and free. If they were not, DSI has quietly replaced their distribution with the Gaussian that shares its mean and covariance.

That substitution has a specific and damaging consequence: **a Gaussian has infinite support in every direction.** It puts probability mass everywhere from $-\infty$ to $+\infty$. Nothing in the algebra knows that streamflow stops at zero, or that travel time does. So it strays past the bounds, and the negative flows above are exactly that happening.

Are our outputs Gaussian? Not remotely - and we can measure it. Skewness is zero for any symmetric distribution:

In [ ]:
from scipy.stats import skew

print('how skewed are the training-data marginals? (0 = symmetric)')
for grp in ['gage-1','particle','headwater','trgw-0-3-8']:
    names = [n for n in groups[grp] if n in data.columns]
    sk = np.abs(skew(data[names].values, axis=0))
    print(f'  {grp:12s} mean |skew| = {sk.mean():5.2f}   worst = {sk.max():5.2f}')

Strongly skewed, all of them - which is entirely normal for groundwater model outputs. Fluxes and travel times are bounded, and have long tails; that is what they *are*.

## Transformations: getting closer to normal

We cannot make the outputs Gaussian. But we can history match a *transformed* version of them that is much closer to Gaussian, and then map back. If $g$ is monotonic and invertible, doing DSI on $g(\mathbf{d})$ and reporting $g^{-1}$ of the result is a legitimate change of variables - the SVD, the standard-normal prior on $\mathbf{x}$ and the linear-Gaussian update all happen in transformed space, where the assumption is much less of a lie.

Two transforms do most of the work, and they address different problems.

**`log10` - for quantities you believe are log-distributed.** The log of a lognormal variable is *exactly* normal, and plenty of hydrologic quantities are roughly lognormal. pyemu's implementation is forgiving about the mechanics: if a column contains zeros or negatives it computes an offset automatically, shifts the data up, logs it, and subtracts the offset again on the way back. So it will never throw an error at you.

That forgiveness is precisely the trap. The transform runs whatever you feed it, but it is still *asserting* that the quantity is log-distributed. If that is wrong, you have not fixed the normality problem, you have replaced it with a different wrong assumption - silently. Note too that the positivity guarantee people reach for only holds when no offset was needed: with a shift applied, the inverse is $10^x - \text{offset}$, which can return negative values again.

**`normal_score` - for anything at all.** Rank every value of an output within its own ensemble, then map that rank to the matching quantile of a standard normal. By construction the transformed marginal is *exactly* $N(0,1)$ - whatever shape it started with: skewed, bounded, bimodal, doesn't matter. No distributional assumption is being smuggled in. The inverse maps back through the empirical distribution, so returned values stay inside the range the training ensemble spanned, which gives physical feasibility for free. That same property is a limitation: it deliberately refuses to extrapolate, unless you ask it to with `quadratic_extrapolation=True`.

It also fixes the variance-scale problem we found earlier. Once every output has been mapped to $N(0,1)$ they all have unit variance, so no group can dominate the SVD by virtue of its units. Heads get a say again - and we will come back and look at what that does to the singular value spectrum, because the result is not what you would guess.

### So pick the transform for the physics, not for convenience

Because `log10` will not complain, the burden is on us. Let's look at what these quantities actually are:

In [ ]:
for grp in ['gage-1','headwater','tailwater','particle']:
    names = [n for n in groups[grp] if n in data.columns]
    v = data[names].values
    print(f'{grp:11s} min = {v.min():12,.1f}   values <= 0: {int((v<=0).sum()):5d} of {v.size}')

- `part_time` (particle travel time) is strictly positive, and a travel time plausibly *is* log-distributed - a mixture of path lengths and velocities multiplying together. `log10` is a defensible choice, and since no offset is needed here the inverse transform genuinely cannot return a negative travel time.
- `gage-1` (streamflow) is non-negative but hits **exactly zero** - the river dries up in some realisations. `log10` would apply an offset and carry on, but a distribution with an atom at zero is not lognormal, so we would be asserting something false.
- `headwater` and `tailwater` are **negative most of the time**, and rightly so: they are groundwater/surface-water *exchange* fluxes, where the sign tells you which way the water moves. Log-transforming a signed quantity is a category error, not a tuning choice. pyemu would shift and log it without a murmur.

So: `log10` on the travel time, `normal_score` on everything else. Transforms are applied in the order listed.

In [ ]:
transforms = [
    # strictly positive and skewed - and log10 guarantees it comes back positive
    {'type':'log10', 'columns':['part_time']},
    # everything else, whatever its shape or sign. Also puts every output on the
    # same footing, so heads are no longer swamped by flows in the SVD
    {'type':'normal_score'},
]

Now rebuild and re-condition. These are the same steps we just worked through, wrapped in a function so we can run them again.

In [ ]:
def build_and_condition(training_data, transforms, tag, noptmax=3):
    """fit a DSI emulator to `training_data`, build a PEST interface for it, and
    condition it - exactly the steps we did by hand above"""
    dsi = DSI(pst=pst, data=training_data, transforms=transforms, energy_threshold=1.0)
    dsi.fit()

    t_d = f'template_{tag}'
    p = dsi.prepare_pestpp(t_d=t_d, use_runstor=True)
    p.pestpp_options['forecasts'] = pst.pestpp_options['forecasts']
    hbd.prep_bins(t_d)
    p.pestpp_options['ies_drop_conflicts'] = True
    p.pestpp_options['ies_num_reals'] = training_data.shape[0]
    p.control_data.noptmax = noptmax
    p.write(os.path.join(t_d,'dsi.pst'), version=2)

    m_d = f'master_dsi_{tag}'
    if os.path.exists(m_d):
        shutil.rmtree(m_d)
    shutil.copytree(t_d, m_d)
    pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=m_d)
    return dsi, m_d


dsi_tx, m_d_tx = build_and_condition(data, transforms, 'tx')
pst_tx = pyemu.Pst(os.path.join(m_d_tx,'dsi.pst'))
oe_pr_tx, oe_pt_tx = hbd.load_ies_obs_ensembles(m_d_tx, case='dsi')

And the question that started this section:

In [ ]:
check_feasible(oe_pt_dsi, 'untransformed')
print()
check_feasible(oe_pt_tx, 'transformed')

Gone. Not reduced - gone, and gone by construction rather than by good fortune. The lowest streamflow the transformed emulator can produce is the lowest one in the training ensemble, because that is what the inverse normal-score transform maps back to.

Let's also check we did not wreck the fit to get here:

In [ ]:
for tag, m_d in [('untransformed','master_dsi'), ('transformed', m_d_tx)]:
    phi = pd.read_csv(os.path.join(m_d,'dsi.phi.actual.csv'))
    print(f'{tag:14s} phi: {phi["mean"].iloc[0]:8.1f} -> {phi["mean"].iloc[-1]:6.2f}')

Both fit the data comfortably. The difference is that one of them produced answers that could actually happen.

Here are the forecasts from the transformed emulator - the ones we should have been looking at all along:

### What the transform did to the emulator itself

Worth going back to the singular value spectrum we looked at when we first fitted the emulator, and putting the transformed version next to it. The singular values are scaled to the first one in each case, since the transformed data has been standardised and the raw absolute magnitudes are no longer comparable.

In [ ]:
ce_untransformed = np.cumsum(dsi.s**2) / np.sum(dsi.s**2)
ce_transformed   = np.cumsum(dsi_tx.s**2) / np.sum(dsi_tx.s**2)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

axes[0].plot(dsi.s / dsi.s[0], color='0.5', label='untransformed')
axes[0].plot(dsi_tx.s / dsi_tx.s[0], 'b-', label='transformed')
axes[0].set_yscale('log')
axes[0].set_xlim(0, 60)
axes[0].set_xlabel('component')
axes[0].set_ylabel('singular value\n(scaled to the first)')
axes[0].set_title('Singular value spectrum')
axes[0].legend(fontsize=8)

axes[1].plot(np.arange(1,len(ce_untransformed)+1), ce_untransformed, color='0.5',
             label='untransformed')
axes[1].plot(np.arange(1,len(ce_transformed)+1), ce_transformed, 'b-',
             label='transformed')
axes[1].axhline(0.99, color='r', ls='--', lw=1.0, label='99% of variance')
axes[1].set_xlim(0, 60)
axes[1].set_xlabel('number of components')
axes[1].set_ylabel('cumulative share of variance')
axes[1].set_title('Cumulative energy')
axes[1].legend(fontsize=8)

plt.tight_layout()

for lbl, c in [('untransformed', ce_untransformed), ('transformed', ce_transformed)]:
    print(f'{lbl:14s}: {int(np.argmax(c>=0.99)+1):3d} components for 99% of the variance')

This is the most surprising figure in the notebook. Transforming made the problem look **harder**: 99% of the variance now takes about 30 components where before it took 6.

That is the right way round, and it is worth understanding why. The raw ensemble was not really low-dimensional - it only looked that way. Almost all of its absolute variance sat in a handful of large-magnitude outputs, so six components could account for 99% of it while barely describing the 325 head columns at all. The spectrum fell off a cliff not because there was no structure left, but because what remained was numerically tiny.

After the normal-score transform every output has unit variance, so "99% of the variance" now means 99% of the variability across *all* the outputs, heads included. It takes 30 components to get there. That number is the honest one - this is what we meant earlier by heads getting a say again.

Notice also that the leading component is *stronger* after transforming (about half the variance, against 38% before). That is a genuine common mode running through the standardised outputs, and it was previously masked by the scale differences. So the transformed spectrum is not simply flatter - it has a clearer dominant signal *and* a much longer tail.

There is a cost, and it lands squarely on the next section. We have 225 realisations, so 225 directions available. Needing 30 components instead of 6 means spending rather more of that budget - and how much budget we have is set by how many times we could afford to run the model.

In [ ]:
for forecast in pst_tx.forecast_names:
    fig,ax = plt.subplots(1,1,figsize=(5,2.8))
    oe_pr_tx.loc[:,forecast].hist(bins=20,alpha=0.5,color="0.5",ax=ax,label='prior')
    oe_pt_tx.loc[:,forecast].hist(bins=20,alpha=0.5,color="b",ax=ax,label='posterior')
    v = pst_tx.observation_data.loc[forecast,"obsval"]
    ax.axvline(v,color='k',ls='--',lw=2.0,label='truth')
    ax.set_ylabel('count')
    ax.set_title(f'{forecast}  (transformed)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

> **One honest caveat.** A normal-score transform makes each output's *marginal* distribution exactly normal. It does **not** make the *joint* distribution multivariate normal - that would need the dependence between outputs to be linear too, and it generally is not. So transforms get us much closer to the assumption DSI needs, and they buy us physical feasibility outright, but they do not make the assumption true. Nonlinear relationships between observations and forecasts still get smeared.

### How does that compare to iES?

Now a comparison, but we need to be careful about what it can and cannot tell us.

The emulator is a **model of the model**. It was built entirely from Freyberg model runs, so it inherits every error the Freyberg model has - and adds approximation error of its own. It cannot be *more* right than the model it was trained on. If the model is wrong about how streamflow responds to recharge, the emulator will be wrong in the same way, and confidently so.

So this is not a contest with a winner. The iES posterior from the previous notebook came from the full-order model - the thing the emulator is approximating. That makes iES the **reference**, and the question is one-directional:

> does the emulator reproduce what the full-order model told us?

Agreement means the emulator is doing its job and we can trust it for the price. Disagreement means the emulator is failing to capture something - or that the two runs used different ensemble sizes and we are looking at sampling noise. What disagreement never means is that DSI found a better answer than the model.

In [ ]:
# the iES posterior from the previous notebook - load_ies_obs_ensembles reads
# whichever iteration PESTPP-IES actually finished on, rather than assuming
ies_d1 = os.path.join('..','part1_13_basic_ies','master_ies1')
_, oe_pt_ies = hbd.load_ies_obs_ensembles(ies_d1, case='freyberg_pp')

fig,axes = plt.subplots(1,len(pst_tx.forecast_names),
                        figsize=(4*len(pst_tx.forecast_names),3.2))
for ax,forecast in zip(axes,pst_tx.forecast_names):
    ax.hist(oe_pt_ies.loc[:,forecast].values,bins=15,alpha=0.5,
            color='g',density=True,label='iES posterior')
    ax.hist(oe_pt_tx.loc[:,forecast].values,bins=15,alpha=0.5,
            color='b',density=True,label='DSI posterior')
    v = pst_dsi.observation_data.loc[forecast,"obsval"]
    ax.axvline(v,color='k',ls='--',lw=2.0,label='truth')
    ax.set_title(forecast)
    ax.set_yticks([])
    ax.legend(fontsize=8)
plt.tight_layout();

Where the two agree, the emulator has faithfully reproduced a full-order-model result for a tiny fraction of the compute - which is the entire proposition, and on a real model it is a very large prize.

Where they disagree, the emulator is the suspect. Bear in mind the two runs used different ensemble sizes (225 against 50), so some of the difference is sampling noise rather than emulator error. Neither of them is a check on whether the *Freyberg model* is right about anything - both inherit whatever it gets wrong, which is why the forecasts still miss the truth in places, exactly as they did in the iES and Monte Carlo notebooks.

## Getting the parameters back

Earlier we promised to come back to this. The emulator equation does not care what the entries of $\mathbf{d}$ represent - it learns a covariance between columns of numbers. So if we want the model's parameters in the posterior, we simply stack them onto the output vector and let DSI learn their covariance too.

The prior parameter ensemble is sitting right next to the observation ensemble we have been using, in the same iES master directory, indexed by the same realisation names.

In [ ]:
# the prior PARAMETER ensemble from the same iES run
pe = pd.read_csv(os.path.join(ies_d,'freyberg_pp.0.par.csv'),index_col=0)
pe.columns = [c.lower() for c in pe.columns]
pe = pe.astype(float)

# stack the parameters onto the outputs, keeping only the realisations we screened
augmented = data.join(pe.loc[data.index, pst.adj_par_names], how='inner')
print(f'outputs only     : {data.shape}')
print(f'outputs + params : {augmented.shape}')

Fit an emulator to that instead, and its forward run now returns parameters alongside the outputs:

In [ ]:
dsi_aug = DSI(pst=pst, data=augmented, energy_threshold=1.0)
dsi_aug.fit()

sim = dsi_aug.predict(np.random.normal(0, 1, dsi_aug.s.shape))
print(f'the forward run now returns {sim.shape[0]} values, including parameters:')
print(sim.loc[['rch0','rch1']].to_string())

And `prepare_pestpp` carries them through as (zero-weight) observations, so a conditioned run gives you a posterior parameter ensemble - the same object iES would have given you, obtained without running the model.

In [ ]:
pst_aug = dsi_aug.prepare_pestpp(t_d='template_aug', use_runstor=True)
print(f'observations in the augmented control file: {pst_aug.nobs}')
print(f'parameters carried through as observations : '
      f'{[p for p in pst.adj_par_names if p in pst_aug.obs_names][:4]} ...')

So "DSI throws away the parameters" is a choice, not a property of the method. Whether the *posterior* parameters it gives you are trustworthy is a separate question - they are only as good as the covariance the ensemble could estimate, which brings us to the actual problem.

## The curse of dimensionality

Look at what happened to the latent dimension when we added those 68 parameter columns:

In [ ]:
print(f'outputs only     : {data.shape[1]:3d} columns -> {dsi.s.shape[0]} latent parameters')
print(f'outputs + params : {augmented.shape[1]:3d} columns -> {dsi_aug.s.shape[0]} latent parameters')
print(f'realisations in the training data: {data.shape[0]}')

Nothing. We added 68 columns and the emulator's latent dimension did not move, because it never depended on the number of columns in the first place. $\Delta\mathbf{D}$ has $N_e$ columns, so $\mathbf{C}_d^{1/2}$ has at most $N_e$ useful ones - point 3 from the maths section, arriving.

This is the whole difficulty, and it is not really about DSI. **We are estimating an $N_d \times N_d$ covariance matrix from $N_e$ samples.** Here $N_d = 469$ and $N_e = 225$, so $\mathbf{C}_d$ has 110,000 distinct entries estimated from 225 realisations. It is enormously rank-deficient, and the directions it *does* contain are estimated from a small sample, so some of the correlations in it are not real - they are sampling noise. (If that sounds familiar, it is exactly the problem localisation exists to fix in iES.)

The uncomfortable part is that adding columns is free and adding realisations is not. Columns cost nothing - they are already-computed model outputs. Realisations cost a model run each. So the temptation on a real problem is always to describe more and sample less, which is precisely the wrong direction.

Rather than assert what that costs, let's measure it: refit and re-condition the emulator using progressively smaller slices of the same training ensemble.

In [ ]:
# Averaging over several random subsets per size, because a single subset is a
# coin toss - we want the trend, not one draw. The full ensemble has only one
# possible subset, so it needs no repeats.
N_REPS = 4
rng = np.random.default_rng(1234)

n_full = data.shape[0]
ne_list = sorted({n for n in [25, 50, 100, n_full] if 10 <= n <= n_full})
print(f'training ensemble sizes: {ne_list}, with up to {N_REPS} random subsets each')

rows = []
for ne in ne_list:
    for rep in range(1 if ne == n_full else N_REPS):
        idx = data.index if ne == n_full else rng.choice(data.index, size=ne, replace=False)
        _, m_d_ne = build_and_condition(data.loc[idx, :], transforms, f'ne{ne}r{rep}')
        p_ne = pyemu.Pst(os.path.join(m_d_ne,'dsi.pst'))
        _, pt_ne = hbd.load_ies_obs_ensembles(m_d_ne, case='dsi')
        for fore in p_ne.forecast_names:
            truth = p_ne.observation_data.loc[fore,'obsval']
            rows.append({'Ne': ne, 'rep': rep, 'forecast': fore,
                         'posterior_sd': pt_ne[fore].std(),
                         'brackets_truth': bool(pt_ne[fore].min() <= truth <= pt_ne[fore].max())})
sweep = pd.DataFrame(rows)
print(f'\n{len(sweep)//len(ne_list)} forecast results per size, '
      f'{sweep.rep.nunique()} repeats at the smaller sizes')

Two things to look at, side by side. On the left, how wide is the posterior forecast distribution - expressed relative to the full-ensemble result, so that forecasts in completely different units can share an axis. On the right, how often does that posterior actually contain the true value.

In [ ]:
# spread relative to the full-ensemble answer, so all four forecasts fit on one axis
reference = sweep.loc[sweep.Ne == n_full].groupby('forecast').posterior_sd.mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for fore, grp in sweep.groupby('forecast'):
    sd = grp.groupby('Ne').posterior_sd
    mean, lo, hi = sd.mean()/reference[fore], sd.min()/reference[fore], sd.max()/reference[fore]
    line, = axes[0].plot(mean.index, mean.values, 'o-', label=fore)
    # the band is the spread over random subsets - i.e. how much luck was involved
    axes[0].fill_between(mean.index, lo.values, hi.values, alpha=0.15, color=line.get_color())

    cov = grp.groupby('Ne').brackets_truth.mean()
    axes[1].plot(cov.index, cov.values, 'o-', label=fore, color=line.get_color())

axes[0].axhline(1.0, color='k', ls='--', lw=1.0)
axes[0].set_xlabel('training ensemble size $N_e$')
axes[0].set_ylabel('posterior spread,\nrelative to the full ensemble')
axes[0].set_title('Sample less, and the posterior\ngets NARROWER')
axes[0].legend(fontsize=8)

axes[1].axhline(1.0, color='k', ls='--', lw=1.0)
axes[1].set_ylim(-0.05, 1.1)
axes[1].set_xlabel('training ensemble size $N_e$')
axes[1].set_ylabel('fraction of repeats whose\nposterior contains the truth')
axes[1].set_title('...and stops containing\nthe truth')
axes[1].legend(fontsize=8)

plt.tight_layout();

The left panel is the one to sit with. **The posterior gets narrower as the training ensemble gets smaller** - every line drops below the dashed reference as you move left. That is backwards. Less information ought to mean *more* uncertainty. But an undersampled covariance matrix cannot see variability it never sampled, so the emulator mistakes a small sample for a well-constrained answer. This is ensemble collapse by undersampling, and it is dangerous precisely because it looks like success: a tight posterior is what we are all hoping for.

The shaded bands are the spread over random subsets of the same size, so they tell you how much of your answer was luck rather than information. They tend to widen as $N_e$ shrinks - for `part_time` the spread between the best and worst subset of 25 is larger than the mean itself. Which subset of 25 runs you happened to have would materially change what you reported.

In [ ]:
# how far does the posterior have to stretch to reach the truth?
fig, axes = plt.subplots(1, len(reference), figsize=(3.4*len(reference), 3.2), squeeze=False)
for ax, fore in zip(axes[0,:], sorted(reference.index)):
    grp = sweep.loc[sweep.forecast == fore]
    truth = pst_tx.observation_data.loc[fore,'obsval']
    for ne, g in grp.groupby('Ne'):
        ax.scatter([ne]*len(g), g.posterior_sd, s=25,
                   c=['b' if b else 'r' for b in g.brackets_truth])
    ax.set_title(fore, fontsize=10)
    ax.set_xlabel('$N_e$')
    ax.set_ylabel('posterior sd')
axes[0,0].scatter([],[],c='b',label='contains truth')
axes[0,0].scatter([],[],c='r',label='misses truth')
axes[0,0].legend(fontsize=8)
fig.suptitle('Every run, coloured by whether its posterior contains the true value', y=1.04)
plt.tight_layout();

Every individual run is on that last figure, red where its posterior missed the truth. The pattern is the point: the misses cluster at small $N_e$, and they are the runs with the *smallest* posterior spread. Narrow and wrong, together.

A modeller who could only afford 25 runs would have reported a confident travel-time forecast that excludes reality, with nothing in the output to suggest a problem.

That is the curse of dimensionality, and it is worth being blunt about how it scales. This toy problem has 68 parameters and we could afford 225 runs, so we are comfortable. A real model might carry $10^4$-$10^6$ parameters and afford a few hundred runs. The arithmetic does not improve: you are always describing a very high-dimensional object with however many directions your run budget bought, and **there is no diagnostic in a DSI run that tells you when you have crossed from "enough" to "not enough"**. The posterior looks equally confident either way.

Practical consequences, none of them exotic:

- Spend your budget on realisations, not on carrying more outputs. Extra columns are nearly free and buy little; extra realisations are expensive and buy the thing you actually need.
- Only carry the outputs you need. Part2's DSI notebook drops most of its observations before fitting for exactly this reason.
- Do the sweep above on your own problem. Halve the ensemble, refit, and see whether the forecasts move. If they do, you were not converged.
- Treat a suspiciously tight posterior as a symptom, not a result.

## Limitations

DSI is fast, it is easy to run, and it is genuinely useful. It is not magic, and the ways it fails are not always loud. In rough order of how often they bite:

**1. It is a model of the model.** The emulator is an approximation of the full-order model, built from a finite sample of its behaviour. Every error in the model is inherited intact, and approximation error is added on top. An emulator can be a faithful stand-in for a bad model - and it will be just as confident. Nothing in a DSI workflow tests whether your model is right.

**2. It will extrapolate, and it will be confidently wrong when it does.** Nothing bounds $\mathbf{x}$, so the emulator can be pushed well outside the region its training runs explored, and it reports no extra uncertainty when that happens - we saw it invent negative streamflow. `normal_score` restricts this by construction, and `ies_drop_conflicts` catches the detectable case (measured values outside the ensemble range), but neither makes extrapolation safe. If you need to go somewhere the ensemble never went, the fix is more model runs.

**3. Garbage in, garbage out - and worse than usual.** Because DSI is built on variance, one blown-up realisation does not just add noise, it can consume the entire latent space. We spent the first third of this notebook on this for good reason. Always look at your training data before you fit.

**4. Untransformed, the normality assumption is not met and it shows.** The reparameterisation reproduces a mean and a covariance, which describes a distribution only if it is Gaussian - and a Gaussian has infinite support, so the emulator produced negative streamflow in 42% of realisations. Separately, an SVD ranks directions by absolute variance and does not know metres from m³/d, so head observations contributed 0.0% of the components until we transformed. Transforms address both. Treat an untransformed DSI run as a first sketch, and choose transforms for what the quantity physically is - `log10` will silently accept a signed flux and assume it is lognormal.

**5. Conditioning stays linear-Gaussian, even with transforms.** `normal_score` makes each output's *marginal* exactly normal; it does not make the *joint* distribution multivariate normal, which would need the dependence between outputs to be linear too. Nonlinear observation-forecast relationships still get smeared. A visible symptom is emulated time series going saw-toothed in the forecast period - smooth where data pins them down, jumping around where it does not. Part2 shows this clearly.

**6. Dimensionality is the real constraint** - see the section above. The emulator gets $N_e$ directions of variability and no more, however many parameters and outputs you stack into $\mathbf{d}$. On this toy problem 225 runs is plenty. On a real model with $10^4$-$10^6$ parameters and a few hundred affordable runs, it is not obviously plenty, and there is no warning when it stops being enough.

**7. It inherits every assumption in your prior.** DSI conditions the prior ensemble; it does not question it. If the prior parameterisation is too coarse - as we found out the hard way in the Monte Carlo and regularisation notebooks - DSI will hand you a posterior that is too narrow, with exactly the same confidence as a good one.

**8. There is no free lunch on the model runs.** DSI moves the cost, it does not remove it. You still need a prior Monte Carlo ensemble big enough to span the behaviour you care about, and that ensemble is the expensive part. What DSI buys you is the ability to condition it - and re-condition it, with different data, different weights, different scenarios - for almost nothing.

## So where does that leave us?

DSI is the first method in these tutorials that decouples the cost of history matching from the cost of running the model. For the Freyberg model that is a curiosity. For a model that takes an hour a run, it is the difference between an uncertainty analysis and no uncertainty analysis.

The catch is that it buys speed with assumptions, and it is quiet about them. Everything in the limitations list above will hold whether or not you check for it.

Next steps:
- [part2_10](../part2_10_eva_and_dsi/2_freyberg_ensemble_data_space_inversion.ipynb) does DSI on the high-dimensional part2 problem, with transformations, noise-based weights and time-series diagnostics
- the same folder has notebooks on ensemble data worth and on PLS emulation, for comparison